# 01 – Business & Data Understanding

**Fase CRISP-DM:** Business Understanding + Data Understanding

## Objetivos
- Consolidar o contexto do negócio e a formulação do problema
- Entender a base (schema, tipos, qualidade e limitações)
- Registrar premissas e riscos (ex.: data leakage)

## Contexto do Negócio
- **Problema**: estimar quanto um cliente tende a gerar de valor ao longo do tempo (LTV) para priorizar canais/produtos e otimizar investimento em aquisição.
- **Tipo de tarefa**: regressão supervisionada (alvo contínuo em R$).
- **Métricas de sucesso**: MAE, RMSE e R².

## Definição do Alvo (LTV) e Premissas
- O dataset já contém a coluna `LTV`.
- Premissa crítica a validar: **qual janela/horizonte** esse LTV representa e **quando** a previsão precisa estar disponível (pré-compra, pós-1ª compra, após X dias etc.).
- Risco de leakage: features diretamente relacionadas ao cálculo do LTV podem criar atalho para o modelo.


## Dicionário de Dados (primeira leitura)

| Coluna | Interpretação provável | Observações |
|---|---|---|
| `ID` | Identificador do cliente | Deve ser único |
| `LTV` | Lifetime Value (R$) | Texto com separador decimal por vírgula |
| `data_compra` | Data/hora da compra | Serial Excel no texto (ex.: `44927,41`) |
| `valor_1_compra` | Valor da 1ª compra (R$) | Texto com separador decimal por vírgula |
| `recorrente_1_compra` | Indicador 0/1 | Verificar se é binário |
| `Produto Fonte` | Produto de entrada | Pode conter ruído de encoding |
| `Fonte Campanha` | Canal/campanha de aquisição | Pode estar desbalanceada |
| `Sexo` | Autodeclaração | Possui `0` como missing mascarado + nulos |
| `Formacao` | Escolaridade | Possui `0` como missing mascarado + nulos |
| `Renda` | Renda mensal (R$) | Muitos zeros e outlier extremo |


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "config" / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import (
    DEFAULT_EXPECTED_COLUMNS,
    clean_raw_data,
    load_config,
    load_raw_data,
    validate_dtypes,
    validate_schema,
)

CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
config = load_config(str(CONFIG_PATH))
RAW_PATH = PROJECT_ROOT / config["paths"]["raw_data"]


## Estrutura da Base (bruta)

In [ ]:
df_raw = load_raw_data(RAW_PATH)
print(f"Linhas: {df_raw.shape[0]} | Colunas: {df_raw.shape[1]}")
validate_schema(df_raw, DEFAULT_EXPECTED_COLUMNS)
display(df_raw.head())


### Tipos, nulos e duplicados (bruto)

In [ ]:
display(df_raw.dtypes)
display(df_raw.isna().sum().sort_values(ascending=False))
print("Duplicados de ID:", int(df_raw["ID"].duplicated().sum()))


## Conversões mínimas para análise

Para permitir EDA e modelagem sem inconsistências, padronizamos:
- `LTV` e `valor_1_compra` → float
- `data_compra` → datetime (a partir do serial Excel)
- `ID` e variáveis categóricas → string; `Renda` e `recorrente_1_compra` → `Int64`
- sentinelas como `"0"` em `Sexo`/`Formacao` → NaN
- normalização de strings com acento corrompido (quando aplicável)


In [ ]:
df, cleaning_report = clean_raw_data(df_raw)
validate_dtypes(df)
display(cleaning_report)
display(df.dtypes)
display(df[["ID", "LTV", "data_compra", "valor_1_compra", "recorrente_1_compra"]].head())


## Principais achados (Data Understanding)
- Tipos originais impedem EDA direta (valores monetários e datas como texto).
- `Sexo` e `Formacao` possuem missing mascarado (`"0"`) além de nulos.
- `Renda` possui grande volume de zeros e um outlier extremo (provável erro de base).
- `Fonte Campanha` aparenta forte concentração em um único valor, com impacto em segmentações.
